<a href="https://colab.research.google.com/github/NayraSousa/mrfi-teste/blob/dev/LENET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install mrfi
!pip install torch
!pip install torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 64.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [3]:
from mrfi import MRFI, EasyConfig
from mrfi.experiment import Acc_experiment, Acc_golden

import torch
import torchvision
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

import math
import random
import csv
import os

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
class LeNet(nn.Module):
  def __init__(self, net_name, input_size, features, trained=False):
    super(LeNet, self).__init__()
    self.conv1 = nn.Conv2d(input_size, 6, 5)
    self.conv2 = nn.Conv2d(6, 16, 5)
    self.fc1 = nn.Linear(features, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 10)

    if trained:
      model = self.load_state_dict(torch.load(f'/content/drive/MyDrive/LeNet/train/lenet_{net_name}.pth'))
      return model
  def forward(self, x):
    x = F.max_pool2d(F.relu(self.conv1(x)), (2,2))
    x = F.max_pool2d(F.relu(self.conv2(x)), 2)
    x = x.view(x.size()[0], -1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

def fit(net_name, epochs=3):
  lenet.to(device)

  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(lenet.parameters(), lr=0.001)

  lenet.train()

  for epoch in range(epochs):
    running_loss = 0
    batch_size = 100
    for i, data in trainloader:
      inputs, labels = i.to(device), data.to(device)

      optimizer.zero_grad()

      outputs = lenet(inputs)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()
      running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(trainloader):.4f}")
  torch.save(lenet.state_dict(), f'/content/drive/MyDrive/LeNet/train/lenet_{net_name}.pth')

  return lenet

def test(lenet, net_name, input_size):
  lenet.load_state_dict(torch.load(f'/content/drive/MyDrive/LeNet/train/lenet_{net_name}.pth'))
  lenet.eval()

  correct = 0
  total = 0

  with torch.no_grad():
      for images, labels in testloader:
          images, labels = images.to(device), labels.to(device)
          outputs = lenet(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()
          break

      print('Accuracy of the network on the 10000 test images: %d %%' % (
        100 * correct / total))

def get_mnist(batch_size=64):
  transform = transforms.Compose(
      [transforms.ToTensor(),
       transforms.Normalize((0.1307,), (0.3081,))]
  )

  trainset = torchvision.datasets.MNIST(
      root='/content/drive/MyDrive/LeNet/data', train=True, download=True, transform=transform
  )
  testset = torchvision.datasets.MNIST(
      root='/content/drive/MyDrive/LeNet/data', train=False, download=True, transform=transform
  )

  trainloader = torch.utils.data.DataLoader(
      trainset, batch_size=batch_size, shuffle=True
  )
  testloader = torch.utils.data.DataLoader(
      testset, batch_size=batch_size, shuffle=False
  )

  return trainloader, testloader, testset

def get_cifar10(batch_size=64):
  transform = transforms.Compose([
      transforms.ToTensor(),
      transforms.Resize((32, 32))
  ])

  trainset = torchvision.datasets.CIFAR10(
      root='/content/drive/MyDrive/LeNet/data', train=True, download=True, transform=transform
  )

  testset = torchvision.datasets.CIFAR10(
      root='/content/drive/MyDrive/LeNet/data', train=False, download=True, transform=transform
  )

  trainloader = torch.utils.data.DataLoader(
      trainset, batch_size=batch_size, shuffle=True
  )
  testloader = torch.utils.data.DataLoader(
      testset, batch_size=batch_size, shuffle=False
  )

  return trainloader, testloader, testset

def load_dataset(dataset='mnist', batch_size=64):

  if dataset.lower()=='mnist':
    train, test, testset = get_mnist()

    return train, test, testset

  if dataset.lower()=='cifar10':
    train, test, testset = get_cifar10()

    return train, test, testset

In [5]:
def calculates_number_positions(e, N, t, p=0.5):

  denominador = 1 + (e ** 2) * ((N-1) / ((t**2*p*(1-p))))
  N_inj = N / denominador

  return N_inj

def return_abs_value(model, layer_name, position):
  layer = getattr(model, layer_name)
  flattened_weights = layer.weight.data.flatten()
  magnitude = torch.abs(flattened_weights[position]).item()

  return magnitude

def return_fi_model(model, layer_name, position, n):

  if layer_name == 'conv1':
    config_str = f"""
                    faultinject:
                      - type: weights
                        name: [weight]
                        quantization:
                          method: SymmericQuantization
                          bit_width: 8
                          dynamic_range: auto
                        selector:
                          method: FixPosition
                          position: {position}
                        error_mode:
                          method: IntFixedBitFlip
                          bit_width: 8
                          bit: {n}
                        module_name: conv1
                    """
  if layer_name == 'fc3':
    config_str = f"""
                    faultinject:
                        - type: weights
                          name: [weight]
                          quantization:
                            method: SymmericQuantization
                            bit_width: 8
                            dynamic_range: auto
                          selector:
                            method: FixPosition
                            position: {position}
                          error_mode:
                            method: IntFixedBitFlip
                            bit_width: 8
                            bit: {n}
                          module_name: fc3
                          """
  econfig = EasyConfig.load_string(config_str)
  fi_model = MRFI(lenet.eval(), econfig)
  fi_model.to(device)

  return fi_model, econfig

def return_calculation_times(model, layer_name, input_size=28):
  if layer_name == 'conv1':
    OFW = ((input_size + 2*lenet.conv1.padding[0] - lenet.conv1.kernel_size[0]) // lenet.conv1.stride[0]) + 1
    OFH = OFW
    CT_i = OFW * OFH

    return CT_i

  if layer_name == 'fc3':
    return 1


def return_new_train(model, trainloader, device=device):
  model.train()
  model.to(device)
  model.zero_grad()
  criterion = nn.CrossEntropyLoss()

  for images, labels in trainloader:
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()

  return model

def return_gradient_value(model, layer_name):

  layer = getattr(model, layer_name)

  return layer.weight.grad.data.flatten()

In [6]:
def evaluate_with_injection(fi_model, loader):
    fi_model.eval()
    correct_gold = 0
    correct_inj  = 0
    total = 0

    fi_model.observers_reset()

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        total += labels.size(0)

        with fi_model.golden_run():
            out_golden = fi_model(images)
            _, pred_g = out_golden.max(dim=1)
            correct_gold += (pred_g == labels).sum().item()

        out_inject = fi_model(images)
        _, pred_f = out_inject.max(dim=1)
        correct_inj += (pred_f == labels).sum().item()

    acc_golden = correct_gold / total
    acc_inject = correct_inj  / total

    obs_res = fi_model.observers_result()

    return acc_golden, acc_inject

In [ ]:
trainloader, testloader,testset = load_dataset('cifar10')

lenet = LeNet('cifar10', 3, 16*5*5)
lenet = fit('cifar10', 1000)
# grad_letnet = return_new_train(lenet, trainloader)
test(lenet, 'cifar10', 3)

for name, param in lenet.named_parameters():
    print(f"Camada: {name}")
    print(f"Parâmetros: {param.count_nonzero()}")
    print("-" * 50)

Epoch 1/1000, Loss: 1.8012
Epoch 2/1000, Loss: 1.5107
Epoch 3/1000, Loss: 1.3894
Epoch 4/1000, Loss: 1.2984
Epoch 5/1000, Loss: 1.2315
Epoch 6/1000, Loss: 1.1843
Epoch 7/1000, Loss: 1.1405
Epoch 8/1000, Loss: 1.1041
Epoch 9/1000, Loss: 1.0749
Epoch 10/1000, Loss: 1.0493
Epoch 11/1000, Loss: 1.0179
Epoch 12/1000, Loss: 0.9907
Epoch 13/1000, Loss: 0.9697
Epoch 14/1000, Loss: 0.9530
Epoch 15/1000, Loss: 0.9294
Epoch 16/1000, Loss: 0.9148
Epoch 17/1000, Loss: 0.8962
Epoch 18/1000, Loss: 0.8726
Epoch 19/1000, Loss: 0.8715
Epoch 20/1000, Loss: 0.8488
Epoch 21/1000, Loss: 0.8368
Epoch 22/1000, Loss: 0.8326
Epoch 23/1000, Loss: 0.8080
Epoch 24/1000, Loss: 0.7965
Epoch 25/1000, Loss: 0.7873
Epoch 26/1000, Loss: 0.7734
Epoch 27/1000, Loss: 0.7662
Epoch 28/1000, Loss: 0.7521
Epoch 29/1000, Loss: 0.7407
Epoch 30/1000, Loss: 0.7316
Epoch 31/1000, Loss: 0.7161
Epoch 32/1000, Loss: 0.7075
Epoch 33/1000, Loss: 0.7032
Epoch 34/1000, Loss: 0.6871
Epoch 35/1000, Loss: 0.6842
Epoch 36/1000, Loss: 0.6728
E

In [ ]:
n_conv1 = calculates_number_positions(0.05, 150, 2.60)
n_fc3 = calculates_number_positions(0.05, 840, 2.60)

pos_conv1 = random.sample(range(150), int(n_conv1))
print(pos_conv1)

pos_fc3 = random.sample(range(840), int(n_fc3))
print(pos_fc3)

In [ ]:
bits = [7]
layers = ['conv1']
metadata = []

CT_conv1 = return_calculation_times(lenet, layers[0])
# CT_fc3 = return_calculation_times(lenet, layers[1])

gradient_conv1 = return_gradient_value(lenet, layers[0])
# gradient_fc3 = return_gradient_value(letnet, layers[1])

In [ ]:
for n in bits:
  for name in layers:

    if name == 'conv1':
      for position in pos_conv1:

        lenet = LeNet(trained=True)
        magnitude = return_abs_value(lenet, name, position)

        fi_model, econfig = return_fi_model(lenet, name, position, n)
        acc_g, acc_f = evaluate_with_injection(fi_model, testloader)

        metadata.append({f'layer_name': name, 'bit': econfig.faultinject[0]['error_mode']['args']['bit'],
                         'parameter': econfig.faultinject[0]['selector']['args']['position'], 'magnitude': magnitude,
                         'vulnerability': acc_g-acc_f, 'calculation_times': CT_conv1,
                         'gradient': gradient_conv1[position].item()})

    # if name == 'fc3':
    #   for position in pos_fc3:

    #     lenet = LeNet(trained=True)
    #     magnitude = return_abs_value(lenet, name, position)

    #     fi_model, econfig = return_fi_model(lenet, name, position, n)
    #     acc_g, acc_f = evaluate_with_injection(fi_model, testloader)

    #     metadata.append({f'layer_name': name, 'bit': econfig.faultinject[0]['error_mode']['args']['bit'],
    #                      'parameter': econfig.faultinject[0]['selector']['args']['position'], 'magnitude': magnitude,
    #                      'vulnerability': acc_g-acc_f, 'calculation_times': CT_fc3,
    #                      'gradient': gradient_fc3[position].item()})

In [ ]:
file_path = '/content/drive/MyDrive/LeNet/teste.csv'
write_header = not os.path.exists(file_path)

with open(file_path, 'a', newline='', encoding='utf-8') as file:
    escritor = csv.DictWriter(file, fieldnames=metadata[0].keys())

    if write_header:
        escritor.writeheader()
    escritor.writerows(metadata)

In [ ]:
# lenet = LeNet(trained=True)
# config_str = f"""
#                     faultinject:
#                         - type: weights
#                           name: [weight]
#                           quantization:
#                             method: SymmericQuantization
#                             bit_width: 8
#                             dynamic_range: auto
#                           selector:
#                             method: FixPosition
#                             position: 190
#                           error_mode:
#                             method: IntFixedBitFlip
#                             bit_width: 8
#                             bit: 7
#                           module_name: fc3
#                           """

# econfig = EasyConfig.load_string(config_str)
# fi_model = MRFI(lenet.eval(), econfig)
# fi_model.to(device)

# acc_g, acc_f = evaluate_with_injection(fi_model, testloader)